In [ ]:
!git clone https://github.com/YPolina/Medicine.git
%cd ./Medicine/BELKA/training
!pip install -r ../requirements.txt
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from lightgbm import LGBMClassifier
import pickle 
import os
import tqdm

In [ ]:
fpg = Chem.dFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024, includeChirality=True)

def compute_fps(smiles_batch):

    """
    Compute fingerprints for a batch of SMILES
    """
    results = []
    for smiles in smiles_batch:
        mol = Chem.MolFromSmiles(smiles.replace('[Dy]', '[H]'))
        if mol is None:
            results.append((np.zeros(1024, dtype=np.int8), {}))
        else:
            fp = fpg.GetCountFingerprint(mol)
            arr = np.zeros(1024, dtype=np.int8)
            Chem.DataStructs.ConvertToNumpyArray(fp, arr)
            results.append(arr)
    return np.array(results)

def train_model_for_protein(protein_data, batch_size=100000, lgb_params=None, eval_set=None):
    """
    Train a LightGBM model for a protein using batched data

    Args:
        protein_data (pd.DataFrame): Data containing SMILES strings and labels
        batch_size (int): Number of samples per batch
        lgb_params (dict): Parameters for the LightGBM classifier
        eval_set (pd.DataFrame): Evaluation set for early stopping

    Returns:
        LGBMClassifier: Trained LightGBM model with the best iteration
    """
    #Split data into batches
    smiles_batches = np.array_split(protein_data['molecule_smiles'].tolist(), -(-len(protein_data) // batch_size))
    label_batches = np.array_split(protein_data['binds'].tolist(), -(-len(protein_data) // batch_size))

    #Prepare evaluation set
    if eval_set is not None and not eval_set.empty:
        eval_features = compute_fps(eval_set['molecule_smiles'])
        eval_labels = eval_set['binds']
        eval_set_lgb = [(eval_features, eval_labels)]
    else:
        eval_set_lgb = None

    #Initialize the LightGBM
    lgb_cls = LGBMClassifier(**lgb_params)

    #Train the model in batches
    for smiles_batch, label_batch in tqdm(zip(smiles_batches, label_batches), total=len(smiles_batches), desc=f"Training {protein_data['protein_name'].unique()[0]}"):
        if len(smiles_batch) != len(label_batch):
            continue

        #Compute fingerprints for the batch
        X_batch = compute_fps(smiles_batch)
        y_batch = np.array(label_batch)

        #Train
        lgb_cls.fit(
            X_batch, 
            y_batch, 
            eval_set=eval_set_lgb, 
            eval_metric='auc', 
            early_stopping_rounds=10, 
            init_model=lgb_cls.booster_ if hasattr(lgb_cls, "booster_") else None
        )

    best_iteration = lgb_cls.best_iteration_
    print(f"Best iteration: {best_iteration}")

    lgb_cls.set_params(n_estimators=best_iteration)

    return lgb_cls

def save_models(model, protein, save_dir="./checkpoints"):
    os.makedirs(save_dir, exist_ok=True)

    model_path = os.path.join(save_dir, f"{protein}_lightgbm.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"Model for {protein} saved at {model_path}")

def train_models_by_protein(train_df, batch_size=100000, save_dir="./checkpoints", lgb_params = None, eval_set = None):
    all_protein_names = train_df['protein_name'].unique()

    for protein in tqdm(all_protein_names, desc="Training models"):
        protein_data = train_df[train_df['protein_name'] == protein]
        if eval_set:
            eval_set = eval_set.copy()
            eval_set = eval_set[eval_set['protein_name'] == protein]
        
        model = train_model_for_protein(protein_data, batch_size, lgb_params, eval_set)

        
        save_models(model, protein, save_dir)

    return "Training complete"


In [ ]:
lgb_params = {
        'max_depth': 11,
        'bagging_fraction': 0.9,
        'learning_rate': 0.05,
        'colsample_bytree': 1,
        'colsample_bynode': 0.5,
        'lambda_l1': 1,
        'objective': 'binary',
        'lambda_l2': 1.5,
        'num_leaves': 490,
        'min_data_in_leaf': 50,
        'verbose': -1,
        'metric': 'average_precision',
        'device': 'cpu'
    }